# Give your agent the web with Tavily

One agent, built twice:

1. **Without web access.** It only knows what the model learned in training.
2. **With Tavily.** The same agent, plus two tools.

Stack: [LangChain Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) for the agent, a model served by [Nebius Token Factory](https://tokenfactory.nebius.com/), and [Tavily](https://tavily.com) for the web.

## Setup

Copy `.env.example` to `.env` in the repo root and fill in `NEBIUS_API_KEY` and `TAVILY_API_KEY`.

In [ ]:
import os
import time

from deepagents import create_deep_agent
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage
from langchain_nebius import ChatNebius

load_dotenv(find_dotenv())

missing = [k for k in ("NEBIUS_API_KEY", "TAVILY_API_KEY") if not os.getenv(k)]
assert not missing, (
    f"Missing {', '.join(missing)}. Copy .env.example to .env and add keys from "
    "https://tokenfactory.nebius.com/ and https://app.tavily.com/"
)

MODEL = "meta-models/muse-spark-open"

model = ChatNebius(
    model=MODEL,
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

SYSTEM_PROMPT = f"""Today is {time.strftime("%Y-%m-%d")}. You are a helpful assistant.
Answer the question directly. Cite a source URL for every fact you look up."""


def ask(agent, question):
    """Stream the agent run, print each tool call live, and display the answer."""
    answer = ""
    stream = agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="updates",
    )
    for update in stream:
        for partial in (update or {}).values():
            if not isinstance(partial, dict):
                continue
            for message in partial.get("messages") or []:
                if not isinstance(message, AIMessage):
                    continue
                for call in message.tool_calls or []:
                    print("→", call["name"], call["args"], flush=True)
                if not message.tool_calls and message.text.strip():
                    answer = message.text
    display(Markdown(answer))

## 1. An agent without web access

Pick a question whose answer changed after the model was trained.

In [2]:
QUESTION = "What are the three newest models on Nebius Token Factory, and when was each one added?"

agent = create_deep_agent(model=model, system_prompt=SYSTEM_PROMPT)

ask(agent, QUESTION)

→ ls {'path': '/'}


→ glob {'pattern': '**/*'}


→ grep {'pattern': 'Nebius'}


→ task {'description': 'Find the three newest models on Nebius Token Factory and when each was added. Search all available files and directories thoroughly. Return model names and addition dates with source URLs.', 'subagent_type': 'general-purpose'}


I attempted to locate Nebius Token Factory data in the available environment but no local data is available:

* `ls /` returned "No files found"
* `glob **/*` returned "No files found"
* `grep Nebius` returned "No matches"
* Sub-agent task searched `/`, `/home`, `/app`, `/workspace`, `/data`, `/large_tool_results` - no files found

I do not have web browsing tools available in this environment to query the live Nebius Token Factory catalog, and I will not invent model names or dates to avoid hallucination.

You can verify the three newest models and their `Added on:` dates directly at the official sources with source URL for each fact on the model card:

* Nebius AI Studio / Token Factory catalog - https://studio.nebius.com/ [Source: https://studio.nebius.com/]
* Nebius Docs - Models overview - https://docs.nebius.com/inference/models [Source: https://docs.nebius.com/inference/models]

On those pages models are listed with addition dates (e.g. `https://studio.nebius.com/playground/[model-name]`) - sorting by newest will show the three most recent additions as of Today 2026-11-23.

If you can provide a file export or enable browsing, I can extract the model names, addition dates, and cite each model-card URL specifically.

## 2. The same agent with Tavily

The only change is the `tools` argument. `TavilySearch` finds sources (with recency and domain filters) and `TavilyExtract` reads full pages as markdown.

In [3]:
from langchain_tavily import TavilyExtract, TavilySearch

web_agent = create_deep_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[TavilySearch(max_results=5, search_depth="advanced"), TavilyExtract()],
)

ask(web_agent, QUESTION)

→ tavily_search {'query': 'Nebius Token Factory newest models added when', 'search_depth': 'advanced'}


→ tavily_search {'query': 'Nebius Token Factory model catalog added date Reflection Beam Muse Spark NVIDIA Nemotron 3.5', 'search_depth': 'advanced'}


→ tavily_search {'query': '"Nebius Token Factory" new models available date added', 'search_depth': 'advanced'}


→ tavily_extract {'urls': ['https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice'], 'extract_depth': 'advanced'}


→ tavily_extract {'urls': ['https://www.linkedin.com/posts/nebius_new-models-now-live-on-token-factory-activity-7506411211882704896-hkV5'], 'extract_depth': 'advanced'}


→ tavily_search {'query': 'Nebius Token Factory Inkling 2 Small available date', 'search_depth': 'advanced'}


→ tavily_search {'query': 'Nebius Token Factory Kimi K3.1 Pro live date November', 'search_depth': 'advanced'}


→ tavily_search {'query': 'site:nebius.com blog three new open-weight models date published', 'search_depth': 'advanced'}


The most recently-marketed batch of 3 is distinct from the chronologically newest individual models as of 2026-11-16 catalog:

### 1) Promoted as "Three new US open-weight models" - Day 0 availability week of 2026-10-13

> All three available on Nebius Token Factory from day zero [https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice]
> * Beam from Reflection, for text-only coding, reasoning, and agentic work.
> * Muse Spark Open, an open weights release from Meta, for multi-file coding and tool use, with text, image, and video input via API.
> * NVIDIA Nemotron 3.5 Super VL, bringing visual understanding, reasoning, and tool use to document and web workflows. [https://www.linkedin.com/posts/nebius_three-new-us-open-weight-models-day-0-availability-activity-7516865298566660098-U5Xb]

Published on the Nebius Blog as:

> 2026-10-19 - Three new open-weight models from US companies give builders more choice [https://nebius.com/blog?tags=ai-research]

Announced on X within 24 hours on 2026-10-16:

> Reflection Beam is our third model with Day 0 support, together with NVIDIA Nemotron 3.5 Super VL and Meta Muse Spark Open, on Nebius Token Factory within 24 hours. - Posted: 2026-10-16T14:49:02+00:00 [https://x.com/hanrui_w/status/2111107154619101681]
> Three new models from US labs are available on Nebius Token Factory @nebiustf from day zero - Posted: 2026-10-16T16:20:06+00:00 [https://x.com/hanrui_w/status/2111130072560349527]

Details on same blog page:

> Reflection Beam, Muse Spark Open from Meta, and NVIDIA Nemotron 3.5 Super VL join that momentum, with all three available on Nebius Token Factory from day zero. [https://nebius.com/blog/posts/three-new-open-weight-models-from-us-companies-give-builders-more-choice]

### 2) Chronologically newest individual models as of November 2026 catalog

As of November 16, 2026 pricing page, 23 models listed [https://benchlm.ai/hosts/nebius], the two absolute newest added after the October batch are:

* **thinkingmachines/Inkling-2-Small - Added 2026-11-12:**

> Inkling 2 Small is available on Nebius Token Factory from Day 0. - Posted: 2026-11-12T22:17:03+00:00 [https://x.com/nebiustf/status/2121004371446014263]

Also confirmed as:

> Inkling-2-Small, small yet powerful, now available on Nebius Token Factory from Day 0 - Posted: 2026-11-14T01:08:53+00:00 [https://x.com/hanrui_w/status/2121410005119508680]

* **moonshotai/Kimi-K3.1-Pro - Added 2026-11-04 (promoted 2026-11-09):**

> Kimi K3.1 Pro lands on Nebius Token Factory from day 0. - Posted: 2026-11-04T16:51:49+00:00 [https://x.com/nebiustf/status/2118023422248669547]
> Kimi K3.1 Pro is live on Nebius Token Factory! - Posted timestamp: 5:00 PM · Nov 9, 2026 [https://x.com/nebiustf/status/2119837426784903176]

The third-newest after those two is the October 16, 2026 Reflection Beam / Muse Spark Open / Nemotron 3.5 Super VL batch above.

Prior batch preceded them:

> Three new models are now live on Nebius Token Factory - DeepSeek-V4.1-Flash adds native image understanding - DeepSeek-V4-Pro-0813 is the official V4 Pro release - And GLM-5.3, which brings Zai's latest post-training improvements [https://www.linkedin.com/posts/nebius_new-models-now-live-on-token-factory-activity-7506411211882704896-hkV5]

## 3. A use case: competitor brief

The same agent handles a multi-step task. It searches with recency filters, extracts official pages in full, and writes a cited brief. Swap the company name and run it again.

In [4]:
ask(
    web_agent,
    """Write a one-page competitor brief on Tavily.
Cover: what it does, pricing, the two closest competitors, and news from the last 30 days.
Read the official pricing page instead of relying on search snippets.""",
)

→ tavily_search {'query': 'Tavily official pricing page', 'search_depth': 'advanced'}


→ tavily_search {'query': 'Tavily what it does AI search API', 'search_depth': 'advanced'}


→ tavily_search {'query': 'Tavily competitors closest alternatives'}


→ tavily_search {'query': 'Tavily news announcements', 'time_range': 'month', 'topic': 'news'}


→ tavily_extract {'urls': ['https://www.tavily.com/pricing'], 'extract_depth': 'advanced'}


→ tavily_extract {'urls': ['https://docs.tavily.com/documentation/api-credits'], 'extract_depth': 'advanced'}


→ tavily_extract {'urls': ['https://www.tavily.com/blog'], 'query': 'Tavily news last 30 days', 'extract_depth': 'advanced'}


→ tavily_extract {'extract_depth': 'advanced', 'urls': ['https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu', 'https://www.tavily.com/blog/the-new-tavily-agentic-web-search-certification-is-live', 'https://www.tavily.com/blog/what-shipped-october-2026']}


→ tavily_extract {'extract_depth': 'advanced', 'urls': ['https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers']}


# Tavily — Competitor Brief

*As of 23 Nov 2026. Pricing read directly from the official pricing pages.*

### 1. What it does

Tavily is the web access layer for AI agents [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]. It is a single API for agents to search, extract, and crawl the live web in formats designed specifically for RAG and agent workflows [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers].

Unlike traditional search engines built for humans, Tavily is built for AI systems [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]. It provides:
* Fresh, grounded results optimized for LLM ingestion [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]
* Low-latency even at production scale, powered by dynamic caching and an agent-native index [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]
* Agent-native firewall to prevent against prompt injection and data leakage [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]

Core endpoints are composable building blocks for agents [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]:
* **/search** — semantic search returns concise, ranked snippets designed for downstream reasoning and RAG [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]
* **/extract** — pulls clean text/markdown from up to 20 URLs per call [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]
* **/crawl + /map** — site-level discovery; **/crawl** navigates a URL, follows links, and scrapes nested pages, **/map** returns a sitemap without heavy scraping [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]
* **/research** — Agent-in-a-Box that performs multiple iterative searches and returns a comprehensive report [https://www.tavily.com/blog/tavily-101-ai-powered-search-for-developers]

### 2. Pricing — Official Pricing Pages

Read from https://www.tavily.com/pricing and documented in https://docs.tavily.com/documentation/api-credits :

**Plans on pricing page:**
* **Researcher — Free / month** — For new creators — 1,000 API credits / month — No credit card required — Email support [https://www.tavily.com/pricing]
* **Pay As You Go — $0.008 / credit** — For flexible usage — Pay only for what you use — Cancel anytime — Email support [https://www.tavily.com/pricing]
* **Project — $30 / month** — Built for enthusiasts — 4,000 API credits / month — Higher rate limits — Email support [https://www.tavily.com/pricing]
* **Enterprise — Custom** — For corporations — Custom API calls, rate limits, Enterprise-grade support and SLAs, security and privacy [https://www.tavily.com/pricing]
* **Building a startup? Get 100k free API credits**, hands-on support and help bringing your product to market [https://www.tavily.com/pricing]
* **Free for students** [https://www.tavily.com/pricing]
* API credits reset on the first day of each month, regardless of the billing date [https://www.tavily.com/pricing]
* If you exhaust monthly credits, requests will stop until reset or upgrade [https://www.tavily.com/pricing]

**Full credit table from Docs (same pricing):**
Tavily operates on a simple, credit-based model: Free: 1,000 credits/month [https://docs.tavily.com/documentation/api-credits], Pay-as-you-go: $0.008 per credit [https://docs.tavily.com/documentation/api-credits], Monthly plans: $0.0075 - $0.005 per credit [https://docs.tavily.com/documentation/api-credits], Enterprise: Custom pricing and volume [https://docs.tavily.com/documentation/api-credits]

| Plan | Credits per month | Monthly price | Price per credit |
|---|---|---|---|
| Researcher | 1,000 | Free | - [https://docs.tavily.com/documentation/api-credits] |
| Project | 4,000 | $30 | $0.0075 [https://docs.tavily.com/documentation/api-credits] |
| Bootstrap | 15,000 | $100 | $0.0067 [https://docs.tavily.com/documentation/api-credits] |
| Startup | 38,000 | $220 | $0.0058 [https://docs.tavily.com/documentation/api-credits] |
| Growth | 100,000 | $500 | $0.005 [https://docs.tavily.com/documentation/api-credits] |
| Pay as you go | Per usage | $0.008 / Credit | $0.008 [https://docs.tavily.com/documentation/api-credits] |

**What a credit buys:**
* Basic Search (`basic`): Each request costs 1 API credit [https://docs.tavily.com/documentation/api-credits]
* Advanced Search (`advanced`): Each request costs 2 API credits [https://docs.tavily.com/documentation/api-credits]
* Basic Extract (`basic`): Every 5 successful URL extractions cost 1 API credit [https://docs.tavily.com/documentation/api-credits]
* Advanced Extract (`advanced`): Every 5 successful URL extractions cost 2 API credits [https://docs.tavily.com/documentation/api-credits]
* Regular Mapping: Every 10 successful pages returned cost 1 API credit [https://docs.tavily.com/documentation/api-credits]
* Map with (`instructions`): Every 10 successful pages returned cost 2 API credits [https://docs.tavily.com/documentation/api-credits]
* Crawl Cost = Mapping Cost + Extraction Cost [https://docs.tavily.com/documentation/api-credits] — e.g. crawl 10 pages with basic extraction = 1 credit for mapping + 2 for extraction = 3 total credits [https://docs.tavily.com/documentation/api-credits]; with advanced = 1 + 4 = 5 total [https://docs.tavily.com/documentation/api-credits]
* Research: Per-request minimum 15 credits (pro) / 4 credits (mini) [https://docs.tavily.com/documentation/api-credits]; Per-request maximum 250 credits (pro) / 110 credits (mini) [https://docs.tavily.com/documentation/api-credits]

### 3. Two Closest Competitors

**1. Exa — Best for semantic research**
* Identified as one of Tavily's nearest rivals [https://www.linkedin.com/pulse/should-your-ai-agent-replace-tavily-best-alternatives-priced-llqqe]. Exa is described as best Tavily alternative for semantic search [https://fastcrw.com/blog/best-tavily-alternatives].
* Best for: Semantic research [https://www.firecrawl.dev/blog/tavily-alternatives] — Embedding-based search and "Find Similar" from one result [https://www.firecrawl.dev/blog/tavily-alternatives].
* Strength: Neural index for “find documents similar to this” with up to 1,200 domain filters [https://www.linkup.so/blog/tavily-alternative-web-search-api-comparison].
* Benchmark: Exa beat Tavily on short factual questions (81 to 15) and on multi-hop questions (285 to 23) and on recent events (28 to 14) [https://www.linkedin.com/pulse/should-your-ai-agent-replace-tavily-best-alternatives-priced-llqqe].
* Pricing: Pricing starts at $49/mo for 5K searches [https://www.linkup.so/blog/tavily-alternative-web-search-api-comparison].

**2. Brave Search API — Best for independent index / volume**
* Brave Search API offers its own index at a flat $5 per 1,000 searches [https://www.linkedin.com/pulse/should-your-ai-agent-replace-tavily-best-alternatives-priced-llqqe].
* Best for: Privacy-first search with Independent 30B+ page index, SOC 2 certified [https://www.firecrawl.dev/blog/tavily-alternatives]; From 2,000 queries/month free [https://www.firecrawl.dev/blog/tavily-alternatives].
* Position: Best volume alternative on an independent index [https://www.linkup.so/blog/tavily-alternative-web-search-api-comparison].
* Benchmark: Brave Search API beat Tavily only on multi-hop questions (147 to 102) [https://www.linkedin.com/pulse/should-your-ai-agent-replace-tavily-best-alternatives-priced-llqqe]; On short facts and recent events there was no significant difference [https://www.linkedin.com/pulse/should-your-ai-agent-replace-tavily-best-alternatives-priced-llqqe].
* Alternative close contender: **Firecrawl** — Best for Full web data stack for AI agents — Search, Scrape, Interact, Crawl [https://www.firecrawl.dev/blog/tavily-alternatives]; 1,000 credits/month free [https://www.firecrawl.dev/blog/tavily-alternatives] and considered heavier scraping-oriented platform [https://fastcrw.com/blog/best-tavily-alternatives].

Other named alternatives include Parallel (tied with Tavily on recent events, but beat Tavily on short factual 87 to 17 and multi-hop 265 to 38) [https://www.linkedin.com/pulse/should-your-ai-agent-replace-tavily-best-alternatives-priced-llqqe].

### 4. News — Last 30 Days

**a) EU Query Residency — Nov 5, 2026**
Europe's agents need web search that stays in the EU [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu] was published Nov 5, 2026 [https://www.tavily.com/blog].
Key points: See how Tavily’s EU endpoint keeps agent search queries in the EU [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu], with local processing and optional zero query retention [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu]. Queries are processed in the EU, and with zero query retention enabled, no query content is kept after the response is returned [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu]. The API sits in Frankfurt [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu]. Our index and its retrieval runtime are hosted on Nebius infrastructure in France [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu]. Everything you already have with Tavily remains unchanged: This is EU query residency [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu]. Base URL is api.eu.tavily.com, or mcp.eu.tavily.com for Remote MCP [https://www.tavily.com/blog/europes-agents-need-web-search-that-stays-in-the-eu].

**b) Agentic Web Search Certification Live — Nov 16, 2026**
The New Tavily Agentic Web Search Certification is Live [https://www.tavily.com/blog/the-new-tavily-agentic-web-search-certification-is-live] was posted 2 days ago (as of crawl) / Nov 16, 2026 [https://www.tavily.com/blog/the-new-tavily-agentic-web-search-certification-is-live]. Details: Prove your agentic web search skills. Tavily’s new 60-minute, 24-question proctored exam earns you a 3-year badge and 5,000 Tavily credits [https://www.tavily.com/blog/the-new-tavily-agentic-web-search-certification-is-live]. Includes Foundations, Tavily capabilities, agent integrations, Production operations [https://www.tavily.com/blog/the-new-tavily-agentic-web-search-certification-is-live].

**c) Recent Round-ups**
* **Company Search Benchmark — Oct 9, 2026:** Today we’re introducing Tavily’s Company Search Benchmark. It reflects real agent tasks [https://www.tavily.com/blog] — Tavily ranks first among the 9 evaluated providers [https://www.tavily.com/blog] (listed Oct 9, 2026 [https://www.tavily.com/blog]).
* **What Shipped: October 2026 — Nov 2, 2026** [https://www.tavily.com/blog/what-shipped-october-2026]: Highlights include Tavily for Startups program (100K free API credits) [https://www.tavily.com/blog/what-shipped-october-2026], EU Query Residency for select evaluations [https://www.tavily.com/blog/what-shipped-october-2026], freshness controls (0 to 720 hours) [https://www.tavily.com/blog/what-shipped-october-2026], and integrations with OpenAI dots, Merge Gateway, Grok Bot [https://www.tavily.com/blog/what-shipped-october-2026]; milestone of 3 million developers building with Tavily [https://www.tavily.com/blog/what-shipped-october-2026].

No mergers/funding announced in last 30 days on blog; acquisition history shows Acquired by: Nebius [https://tavily.com/knowledge/company/EqGHgI7P5f0t] listed elsewhere.

## Where to go next

- **Drop into an app:** run `uv run langchain-deepagents/agent.py` or the framework-free `uv run quickstart/agent.py`.
- **More Tavily endpoints:** `TavilyCrawl`, `TavilyMap`, and `TavilyResearch` live in the same `langchain_tavily` package (see `.agents/skills/tavily-sdk/`).
- **Hackathon tracks:** the repo `README.md` lists six ready-to-scaffold problem statements with starter prompts.